# MARBERT Fine-tuning: Joint Learning versus Sentiment Only

**Status: not executed in the onboarding CPU environment.** Hugging Face model downloads were denied by the environment network policy. This notebook is ready for a GPU runtime with model access; it is not evidence of completed MARBERT training.

The shared MARBERT encoder feeds two classification heads. Joint loss is sentiment cross-entropy + sarcasm cross-entropy. The control uses sentiment loss alone. Both select their best epoch using validation sentiment macro-F1.

Recommended: a GPU runtime; batch size 8 and sequence length 128 are starting settings, not guaranteed memory requirements. Reduce batch size if necessary. Run the following installation cell in a fresh notebook runtime.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path("pyproject.toml").exists(), "Clone the repository and change into its root directory first."

In [ ]:
%pip install -e ".[train,dev]"

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
os.chdir(ROOT)
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Use a GPU runtime for the complete MARBERT experiment."

## Model revision
Resolve and record an immutable model revision before both runs. Internet access to Hugging Face and its artifact hosts is required. The dataset downloader separately verifies immutable data hashes.

In [ ]:
from huggingface_hub import model_info
REVISION = model_info("UBC-NLP/MARBERT").sha
print("MARBERT revision:", REVISION)

In [ ]:
for task in ["sentiment", "joint"]:
    subprocess.run([sys.executable, "-m", "maqsad.train", "--task", task, "--revision", REVISION,
        "--epochs", "3", "--batch-size", "8", "--seed", "42", "--output", f"artifacts/marbert-{task}"], check=True)

In [ ]:
import pandas as pd
reports = {task: json.loads(Path(f"artifacts/marbert-{task}/metrics.json").read_text()) for task in ["sentiment", "joint"]}
display(pd.DataFrame({task: {"sentiment_macro_f1": report["test"]["sentiment_macro_f1"], "sentiment_accuracy": report["test"]["sentiment_accuracy"]} for task, report in reports.items()}))
print("Joint sarcasm F1:", reports["joint"]["test"]["sarcasm_f1"])

In [ ]:
from maqsad.inference import predict_transformer
predict_transformer(["يا سلام على الخدمة، ساعتين وأنا أنتظر!"])

## Interpretation
A positive difference for a single seed is preliminary. Repeat both tasks with identical seeds 42, 43 and 44 and report mean and standard deviation. Do not tune on the test set. Copy only final aggregate metrics into reports; keep checkpoints and raw tweets out of Git.